# Fundamentals of Scientific Computing — Atomic Toy Model

**Group members**

- Krishna Chaitanya Namala — ES24BTECH11023
- Rohith Perugu — ES24BTECH11026
- Purna Nanda Reddy Chintapally — ES24BTECH11014
- Member 4 — Roll number: **TO BE FILLED**
- Member 5 (if applicable) — Roll number: **TO BE FILLED**
- Member 6 (if applicable) — Roll number: **TO BE FILLED**

This notebook follows the original notebook's simple section-and-code-cell style. It uses basic Python, NumPy and Matplotlib. Run cells from top to bottom.

**Model:** particles move in a 2D square box. Pairs interact through a spring-like force only when their separation is smaller than the cutoff radius.

# Importing required libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import os
import subprocess
import ctypes
import math
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import display, Image

# 1. Setting up the world

In [ ]:
np.random.seed(42)

N = 12
L = 400.0
dt = 0.001
T = 1000

k = 15.0
r0 = 80.0
rc = 150.0

pos = np.random.uniform(0, L, (N, 2))
vel = np.random.uniform(0, 10, (N, 2))
mass = 10.0 * np.ones(N)

print("Number of particles:", N)
print("Box size:", L)
print("Time step:", dt)
print("Number of steps:", T)

# 2. Wall Reflections

In [ ]:
def walls_reflect(pos, vel, L):
    # Reflect particles at the left and right walls
    left_hits = pos[:, 0] < 0
    pos[left_hits, 0] *= -1
    vel[left_hits, 0] *= -1

    right_hits = pos[:, 0] > L
    pos[right_hits, 0] = 2 * L - pos[right_hits, 0]
    vel[right_hits, 0] *= -1

    # Reflect particles at the bottom and top walls
    bottom_hits = pos[:, 1] < 0
    pos[bottom_hits, 1] *= -1
    vel[bottom_hits, 1] *= -1

    top_hits = pos[:, 1] > L
    pos[top_hits, 1] = 2 * L - pos[top_hits, 1]
    vel[top_hits, 1] *= -1

    return pos, vel

# Testing wall reflections

In [ ]:
test_pos = np.array([[-1.0, 50.0], [L + 1.0, 50.0],
                     [50.0, -1.0], [50.0, L + 1.0]])
test_vel = np.array([[-2.0, 0.0], [2.0, 0.0],
                     [0.0, -3.0], [0.0, 4.0]])
test_pos, test_vel = walls_reflect(test_pos, test_vel, L)
print("Positions after reflection:")
print(test_pos)
print("Velocities after reflection:")
print(test_vel)
print("All test particles inside box:", np.all((test_pos >= 0) & (test_pos <= L)))

# 3. Pairwise distances and neighbour identification

In [ ]:
def pairwise_distances(pos):
    displacement = pos[:, None, :] - pos[None, :, :]
    distances = np.linalg.norm(displacement, axis=2)
    return distances

def find_neighbors(distances, rc):
    neighbors = distances < rc
    np.fill_diagonal(neighbors, False)
    return neighbors

distances = pairwise_distances(pos)
neighbors = find_neighbors(distances, rc)
print("Distance matrix shape:", distances.shape)
print("Number of unique neighbour pairs:", np.sum(neighbors) // 2)

# Verification of pairwise distances and neighbours

In [ ]:
print("Distance matrix symmetric:", np.allclose(distances, distances.T))
print("Diagonal distances are zero:", np.allclose(np.diag(distances), 0))
print("No self-neighbours:", not np.any(np.diag(neighbors)))
print("Neighbour matrix symmetric:", np.array_equal(neighbors, neighbors.T))
print("Maximum symmetry error:", np.max(np.abs(distances - distances.T)))

# 4. Pair forces: springs between particles

In [ ]:
def pair_forces(pos):
    forces = np.zeros_like(pos, dtype=float)

    for i in range(N - 1):
        for j in range(i + 1, N):
            r = pos[j] - pos[i]
            distance = np.linalg.norm(r)

            if distance < rc and distance > 1e-12:
                force = k * (distance - r0) * r / distance
                forces[i] += force
                forces[j] -= force

    return forces

forces = pair_forces(pos)
net_force = np.sum(forces, axis=0)
print("Net internal force:", net_force)
print("Magnitude of net internal force:", np.linalg.norm(net_force))

# 5. Kinetic energy

In [ ]:
def kinetic_energy(vel, mass):
    return np.sum(0.5 * mass * np.sum(vel ** 2, axis=1))

print("Kinetic Energy:", kinetic_energy(vel, mass))

# 6. Potential energy

For a pair inside the cutoff, the spring potential is used. We subtract its value at `rc`, so the potential approaches zero continuously when a pair reaches the cutoff. This avoids an artificial jump in potential energy when a pair enters or leaves the interaction range.

In [ ]:
def potential_energy(pos):
    energy = 0.0
    cutoff_energy = 0.5 * k * (rc - r0) ** 2

    for i in range(N - 1):
        for j in range(i + 1, N):
            distance = np.linalg.norm(pos[j] - pos[i])
            if distance < rc:
                energy += 0.5 * k * (distance - r0) ** 2 - cutoff_energy

    return energy

# 7. Total mechanical energy

In [ ]:
def total_energy(pos, vel, mass):
    return kinetic_energy(vel, mass) + potential_energy(pos)

print("Kinetic Energy:", kinetic_energy(vel, mass))
print("Potential Energy:", potential_energy(pos))
print("Total Mechanical Energy:", total_energy(pos, vel, mass))

# 8. Time integration

We use the semi-implicit Euler method: first update velocity using acceleration, then update position using the new velocity.

In [ ]:
def step_once(pos, vel, mass, dt, L):
    force = pair_forces(pos)
    acceleration = force / mass[:, None]
    vel = vel + acceleration * dt
    pos = pos + vel * dt
    pos, vel = walls_reflect(pos, vel, L)
    return pos, vel

# Testing one simulation step

In [ ]:
test_pos = pos.copy()
test_vel = vel.copy()
new_pos, new_vel = step_once(test_pos, test_vel, mass, dt, L)
print("Initial positions (first 3 particles):")
print(np.round(pos[:3], 3))
print("Positions after one step:")
print(np.round(new_pos[:3], 3))
print("All particles inside box:", np.all((new_pos >= 0) & (new_pos <= L)))

# 9. Running the complete conservative simulation

In [ ]:
pos_sim = pos.copy()
vel_sim = vel.copy()
K_history = []
U_history = []
E_history = []
trajectory = []

K_history.append(kinetic_energy(vel_sim, mass))
U_history.append(potential_energy(pos_sim))
E_history.append(K_history[-1] + U_history[-1])
trajectory.append(pos_sim.copy())

for step in range(T):
    pos_sim, vel_sim = step_once(pos_sim, vel_sim, mass, dt, L)
    K_history.append(kinetic_energy(vel_sim, mass))
    U_history.append(potential_energy(pos_sim))
    E_history.append(K_history[-1] + U_history[-1])
    trajectory.append(pos_sim.copy())

K_history = np.array(K_history)
U_history = np.array(U_history)
E_history = np.array(E_history)
trajectory = np.array(trajectory)
time_values = np.arange(T + 1) * dt

print("Simulation completed")
print("Trajectory shape:", trajectory.shape)
print("Initial total energy:", E_history[0])
print("Final total energy:", E_history[-1])
print("Maximum relative energy drift:", np.max(np.abs(E_history - E_history[0])) / abs(E_history[0]))
print("All particles inside box:", np.all((trajectory >= 0) & (trajectory <= L)))

# 10. Energy variation during the simulation

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(time_values, K_history, label="Kinetic Energy")
plt.plot(time_values, U_history, label="Potential Energy")
plt.plot(time_values, E_history, label="Total Energy")
plt.xlabel("Time")
plt.ylabel("Energy")
plt.title("Energy vs Time: Conservative Simulation")
plt.legend()
plt.grid(True)
plt.show()

# 11. Particle trajectories

In [ ]:
plt.figure(figsize=(7, 7))
for i in range(N):
    plt.plot(trajectory[:, i, 0], trajectory[:, i, 1], linewidth=1)
    plt.scatter(trajectory[0, i, 0], trajectory[0, i, 1], s=20)
plt.xlim(0, L)
plt.ylim(0, L)
plt.xlabel("x position")
plt.ylabel("y position")
plt.title("Particle trajectories")
plt.grid(True)
plt.show()

# 12. Final numerical checks for Task 1

In [ ]:
final_forces = pair_forces(pos_sim)
print("Distance symmetry:", np.allclose(distances, distances.T))
print("Zero diagonal:", np.allclose(np.diag(distances), 0))
print("No self-neighbour:", not np.any(np.diag(neighbors)))
print("Net force magnitude:", np.linalg.norm(np.sum(final_forces, axis=0)))
print("Minimum coordinate in trajectory:", np.min(trajectory))
print("Maximum coordinate in trajectory:", np.max(trajectory))
print("All particles stay inside box:", np.all((trajectory >= 0) & (trajectory <= L)))

# 13. Task 2 — Particle animation

The animation needs a longer run than the baseline. The baseline only covers 1 time unit, so at the start particles move less than one pixel per frame and the animation looks jittery. Here we simulate **12 particles for 3000 time steps** with `dt = 0.005` (15 time units), using the same initial positions and velocities and the same `step_once` function. The box is drawn as a thick black square and each particle leaves a short trail so the motion is easy to follow.

In [ ]:
from matplotlib.patches import Rectangle

# Longer run just for the animation: 12 particles, 3000 time steps
anim_dt = 0.005
anim_steps = 3000
p = pos.copy()
v = vel.copy()
anim_traj = [p.copy()]
anim_vel = [v.copy()]
for step in range(anim_steps):
    p, v = step_once(p, v, mass, anim_dt, L)
    anim_traj.append(p.copy())
    anim_vel.append(v.copy())
anim_traj = np.array(anim_traj)
anim_vel = np.array(anim_vel)

skip = 5            # show every 5th step
trail_frames = 15   # how many past frames are drawn as a trail
colors = plt.cm.tab20(np.arange(N) % 20)

fig, ax = plt.subplots(figsize=(5, 5))
ax.add_patch(Rectangle((0, 0), L, L, fill=False, edgecolor="black", linewidth=3))
ax.set_xlim(-20, L + 20)
ax.set_ylim(-20, L + 20)
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])
trails = [ax.plot([], [], color=colors[i], alpha=0.5, linewidth=1.5)[0] for i in range(N)]
dots = ax.scatter(anim_traj[0, :, 0], anim_traj[0, :, 1], s=90, c=colors, edgecolors="black", zorder=3)

def update(step):
    dots.set_offsets(anim_traj[step])
    start = max(0, step - trail_frames * skip)
    for i in range(N):
        trails[i].set_data(anim_traj[start:step + 1:skip, i, 0], anim_traj[start:step + 1:skip, i, 1])
    ax.set_title("Atomic Toy Model: t = " + format(step * anim_dt, ".2f"))
    return []

animation = FuncAnimation(fig, update, frames=range(0, len(anim_traj), skip), interval=33)
animation.save("particle_animation.gif", writer=PillowWriter(fps=30), dpi=80)
plt.close(fig)
print("Animation saved as particle_animation.gif")
display(Image(filename="particle_animation.gif"))

In [ ]:
# Numbers that help us read the animation
frame_moves = np.linalg.norm(np.diff(anim_traj[::skip], axis=0), axis=2)
speeds = np.linalg.norm(anim_vel, axis=2)
closest = min(np.min(np.linalg.norm(a[:, None] - a[None], axis=2) + np.eye(N) * 1e9) for a in anim_traj[::10])
near_wall = ((anim_traj < 5) | (anim_traj > L - 5)).any(axis=2)

print("Number of particles:", N, "  Number of time steps:", anim_steps)
print("All particles inside box:", np.all((anim_traj >= 0) & (anim_traj <= L)))
print("Smallest / largest coordinate:", anim_traj.min(), anim_traj.max())
print("Largest move of one particle between frames:", frame_moves.max(), "(box size", L, ")")
print("Mean speed at start / end:", speeds[0].mean(), speeds[-1].mean(), "  Maximum speed:", speeds.max())
print("Closest distance between two particles:", closest)
print("Largest fraction of time one particle spends within 5 units of a wall:", near_wall.mean(axis=0).max())

**What the animation showed us** (the numbers are printed in the cell above):

- **No particle escaped.** All coordinates stayed between about 0.02 and 399.95, so the wall reflection works.
- **No particle got stuck on a wall.** The most time any one particle spent within 5 units of a wall was about 3% of the run.
- **No unrealistic jumps.** The largest move of a particle between two frames was about 6.5 units on a 400-unit box.
- **No numerical blow-up.** The mean speed grows from about 7 to about 100 because the random start has stretched springs that release their stored energy. This is the K <-> U exchange from Task 3.
- **Clustering is expected.** Particles that start far apart are pulled toward each other (the springs attract when the distance is larger than `r0 = 80`).
- **One model limitation.** Two particles came as close as 1.3 units. The spring repulsion inside `r0` is soft, so two particles can almost overlap. This is a property of the model, not a bug.
- The first version of this animation was jittery only because it covered 1 time unit, so particles moved less than a pixel per frame at the start.

# 14. Task 3 — Energy and timestep accuracy study

We compare the requested time steps over the same physical duration (10 time units, roughly two K <-> U exchange cycles). The same initial positions and velocities are used in every run, and the integration method is the baseline `step_once`. For each dt we plot K, U and E, and the maximum relative energy drift is the error measure.

In [ ]:
def run_conservative(dt_run, total_time):
    steps = int(total_time / dt_run)
    p = pos.copy()
    v = vel.copy()
    K = np.zeros(steps + 1)
    U = np.zeros(steps + 1)
    K[0] = kinetic_energy(v, mass)
    U[0] = potential_energy(p)

    for step in range(steps):
        p, v = step_once(p, v, mass, dt_run, L)
        K[step + 1] = kinetic_energy(v, mass)
        U[step + 1] = potential_energy(p)

    return np.arange(steps + 1) * dt_run, K, U, K + U

timestep_values = [0.05, 0.02, 0.01, 0.005]
total_time_study = 10.0
accuracy_results = {}

print("dt\tmaximum relative energy drift")
for dt_test in timestep_values:
    t, K, U, E = run_conservative(dt_test, total_time_study)
    error = np.max(np.abs(E - E[0])) / abs(E[0])
    accuracy_results[dt_test] = (t, K, U, E, error)
    print(dt_test, "\t", error)

In [ ]:
# K, U and E for every time step
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, dt_test in zip(axes.flatten(), timestep_values):
    t, K, U, E, error = accuracy_results[dt_test]
    ax.plot(t, K, label="K")
    ax.plot(t, U, label="U")
    ax.plot(t, E, label="E = K + U", linewidth=2)
    ax.set_title("dt = " + str(dt_test) + "  (drift = " + format(error, ".2e") + ")")
    ax.set_xlabel("Time")
    ax.set_ylabel("Energy")
    ax.legend()
    ax.grid(True)
plt.tight_layout()
plt.show()

# Total energy for all time steps together
plt.figure(figsize=(10, 5))
for dt_test in timestep_values:
    t, K, U, E, error = accuracy_results[dt_test]
    plt.plot(t, E, label="dt = " + str(dt_test))
plt.xlabel("Time")
plt.ylabel("Total mechanical energy")
plt.title("Total energy for different time steps")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Largest time step whose drift is below our chosen tolerance
tolerance = 0.05      # 5 percent: our choice, explained below
good_dts = [d for d in timestep_values if accuracy_results[d][4] < tolerance]
largest_dt = max(good_dts)
print("Tolerance:", tolerance)
print("Time steps that meet it:", good_dts)
print("Largest defensible time step:", largest_dt, " with drift", accuracy_results[largest_dt][4])

**Answer:** Over 10 time units the energy drift is about 18% at dt = 0.05, 6.3% at 0.02, 3.2% at 0.01 and 1.3% at 0.005, so it shrinks roughly in proportion to dt (the baseline method is first order). K and U trade energy back and forth while E stays nearly flat for the smaller dt. With a tolerance of 5% drift (our choice) the **largest defensible time step is dt = 0.01**. If the group wants a stricter tolerance such as 2%, the answer becomes dt = 0.005. If the numbers on your computer differ, update the values in this paragraph.

# 15. Task 4 — Dissipation and relaxation

A linear drag force is added: `F_drag = -gamma * velocity`. For each gamma we use the same initial condition. Positive gamma removes mechanical energy.

**Steady-state criterion (from the task):** the system has relaxed when the energy change over a 10-time-unit window, `|E(t) - E(t - 10)| / |E(0)|`, stays below 1e-3 until the end of the run. We run each gamma for 300 time units with `dt = 0.01`, which met our energy tolerance in Task 3. If the criterion is not met by then, we report that it was not reached. The cell takes under a minute.

In [ ]:
def total_forces(pos, vel, gamma):
    return pair_forces(pos) - gamma * vel

def step_with_drag(pos, vel, mass, dt, L, gamma):
    force = total_forces(pos, vel, gamma)
    acceleration = force / mass[:, None]
    vel = vel + acceleration * dt
    pos = pos + vel * dt
    pos, vel = walls_reflect(pos, vel, L)
    return pos, vel

def find_relax_time(times, energy, window=10, tol=1e-3):
    # first time from which |E(t) - E(t - 10 time units)| / |E(0)| stays below tol until the end of the run
    change = np.abs(energy[window:] - energy[:-window]) / abs(energy[0])
    for i in range(len(change)):
        if np.all(change[i:] < tol):
            return times[i + window]
    return None

def run_dissipation(gamma, dt_run, max_time, record_every=100):
    p = pos.copy()
    v = vel.copy()
    times = [0.0]
    kinetic = [kinetic_energy(v, mass)]
    potential = [potential_energy(p)]
    speed = [np.mean(np.linalg.norm(v, axis=1))]

    for step in range(1, int(max_time / dt_run) + 1):
        p, v = step_with_drag(p, v, mass, dt_run, L, gamma)
        if step % record_every == 0:
            times.append(step * dt_run)
            kinetic.append(kinetic_energy(v, mass))
            potential.append(potential_energy(p))
            speed.append(np.mean(np.linalg.norm(v, axis=1)))

    times = np.array(times)
    energy = np.array(kinetic) + np.array(potential)
    relax_time = find_relax_time(times, energy) if gamma > 0 else None
    return times, np.array(kinetic), np.array(potential), energy, np.array(speed), relax_time, p

gamma_values = [0.0, 0.05, 0.1, 0.5]
dissipation_results = {}
print("gamma | relaxation time (energy criterion) | final energy | final mean speed")
for gamma in gamma_values:
    max_time = 100.0 if gamma == 0 else 300.0
    result = run_dissipation(gamma, 0.01, max_time)
    dissipation_results[gamma] = result
    t, kinetic, potential, energy, speed, relax_time, final_pos = result
    print(gamma, "|", relax_time, "|", energy[-1], "|", speed[-1])

In [ ]:
# Conservative (gamma = 0) versus dissipative: total energy over the first 100 time units
plt.figure(figsize=(10, 5))
for gamma in gamma_values:
    t, kinetic, potential, energy, speed, relax_time, final_pos = dissipation_results[gamma]
    keep = t <= 100
    plt.plot(t[keep], energy[keep], label="gamma = " + str(gamma))
plt.xlabel("Time")
plt.ylabel("Total mechanical energy")
plt.title("Total energy for different drag values")
plt.legend()
plt.grid(True)
plt.show()

# K, U and E for each gamma
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, gamma in zip(axes.flatten(), gamma_values):
    t, kinetic, potential, energy, speed, relax_time, final_pos = dissipation_results[gamma]
    ax.plot(t, kinetic, label="K")
    ax.plot(t, potential, label="U")
    ax.plot(t, energy, label="E", linewidth=2)
    ax.set_title("K, U and E for gamma = " + str(gamma))
    ax.set_xlabel("Time")
    ax.set_ylabel("Energy")
    ax.legend()
    ax.grid(True)
plt.tight_layout()
plt.show()

# Mean speed versus time (log scale) with the steady-state threshold
plt.figure(figsize=(10, 5))
for gamma in gamma_values:
    t, kinetic, potential, energy, speed, relax_time, final_pos = dissipation_results[gamma]
    plt.semilogy(t, speed, label="gamma = " + str(gamma))
plt.axhline(1e-3, color="black", linestyle="--", label="threshold 1e-3")
plt.xlabel("Time")
plt.ylabel("Mean particle speed")
plt.title("Mean speed for different drag values")
plt.legend()
plt.grid(True)
plt.show()

# Final positions: conservative run versus dissipative runs
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, gamma in zip(axes, gamma_values):
    final_pos = dissipation_results[gamma][6]
    ax.add_patch(Rectangle((0, 0), L, L, fill=False, edgecolor="black", linewidth=2))
    ax.scatter(final_pos[:, 0], final_pos[:, 1], s=50)
    ax.set_xlim(-20, L + 20)
    ax.set_ylim(-20, L + 20)
    ax.set_aspect("equal")
    ax.set_title("Final positions, gamma = " + str(gamma))
plt.tight_layout()
plt.show()

In [ ]:
# Steady-state check and relaxation time
for gamma in gamma_values:
    t, kinetic, potential, energy, speed, relax_time, final_pos = dissipation_results[gamma]
    final_neighbors = find_neighbors(pairwise_distances(final_pos), rc)
    if gamma == 0:
        print("gamma = 0: conservative run, no relaxation expected. Final mean speed =", speed[-1])
    elif relax_time is None:
        print("gamma =", gamma, ": criterion NOT reached by t =", t[-1], " final mean speed =", speed[-1])
    else:
        print("gamma =", gamma, ": relaxed at t =", relax_time,
              " gamma * relax_time =", gamma * relax_time,
              " final energy =", energy[-1],
              " pairs within cutoff =", np.sum(final_neighbors) // 2)

**Answer:** Larger gamma relaxes faster. With gamma = 0.5 the criterion is met at t = 149 and the mean speed has dropped to about 0.04. With gamma = 0.1 and gamma = 0.05 the criterion is **not** met within 300 time units: the system is still moving (mean speed about 21 and 48) and the energy is still falling (about -2.00e6 and -1.81e6, heading toward the gamma = 0.5 value of about -2.04e6). So we cannot claim steady state for those two; they would need longer runs. The conservative run (gamma = 0) never settles: its energy only wobbles around the starting value and its mean speed stays large. Drag removes energy on a time scale of about mass/gamma, so a smaller gamma needs proportionally longer. If the numbers on your computer differ, update the values in this paragraph.

# 16. Task 5 — Numerical precision: float32 vs float64

We run the same initial condition in both data types. We compare final positions, energy behaviour, runtime, and storage required by the position and velocity arrays.

In [ ]:
def run_precision(dtype, dt_run=0.001, steps=1000):
    p = pos.astype(dtype).copy()
    v = vel.astype(dtype).copy()
    m = mass.astype(dtype).copy()
    start = time.perf_counter()

    for step in range(steps):
        # Calculate forces using the same simple pair loop
        f = np.zeros_like(p)
        for i in range(N - 1):
            for j in range(i + 1, N):
                r = p[j] - p[i]
                distance = np.sqrt(np.sum(r * r))
                if distance < rc and distance > 1e-12:
                    force = k * (distance - r0) * r / distance
                    f[i] += force
                    f[j] -= force
        v = v + (f / m[:, None]) * dt_run
        p = p + v * dt_run
        p, v = walls_reflect(p, v, L)

    elapsed = time.perf_counter() - start
    return p, v, elapsed

# Repeat the timing 5 times and use the median
times32 = []
times64 = []
for repeat in range(5):
    pos32, vel32, t32 = run_precision(np.float32)
    pos64, vel64, t64 = run_precision(np.float64)
    times32.append(t32)
    times64.append(t64)
time32 = np.median(times32)
time64 = np.median(times64)

position_error = np.linalg.norm(pos32.astype(float) - pos64) / np.linalg.norm(pos64)
mem32 = pos32.nbytes + vel32.nbytes
mem64 = pos64.nbytes + vel64.nbytes
print("float32 median runtime:", time32)
print("float64 median runtime:", time64)
print("Time ratio float64 / float32:", time64 / time32)
print("Relative final-position difference:", position_error)
print("float32 position + velocity memory (bytes):", mem32)
print("float64 position + velocity memory (bytes):", mem64)
print("Memory ratio float64 / float32:", mem64 / mem32)

initial_energy_precision = total_energy(pos.astype(float), vel.astype(float), mass)
energy32_final = total_energy(pos32.astype(float), vel32.astype(float), mass)
energy64_final = total_energy(pos64.astype(float), vel64.astype(float), mass)
print("float32 final mechanical energy:", energy32_final)
print("float64 final mechanical energy:", energy64_final)
print("float32 relative energy change:", abs(energy32_final - initial_energy_precision) / abs(initial_energy_precision))
print("float64 relative energy change:", abs(energy64_final - initial_energy_precision) / abs(initial_energy_precision))

**Answer:** float32 is accurate enough for this short test. After 1000 steps the final positions differ by only about 7e-7 (relative), and the relative energy change is the same as in float64 to three digits. The clear advantage is memory: float32 uses half the bytes (ratio 2.0). The speed advantage here is essentially none (time ratio about 1.0), because with 12 particles and a Python loop the run time is dominated by Python overhead, not by the size of the numbers. float32 matters for large N: for N = 10000 the pair-displacement array is 1.5 GB in float64 but 0.76 GB in float32. Caveat: this only tests 1 time unit, and float32 errors could grow in a longer run.

# 17. Task 6 — Efficiency: Python loop, NumPy, and compiled C

The three force calculations below use the same random particle positions, spring parameters, and cutoff for each N. Each timing is repeated five times and the median is reported. The benchmark box is larger than the physical simulation box so the number of neighbours remains relatively small.

In [ ]:
def force_python_loop(p, k_value, r0_value, rc_value):
    n = len(p)
    f = np.zeros_like(p)
    for i in range(n - 1):
        for j in range(i + 1, n):
            dx = p[j, 0] - p[i, 0]
            dy = p[j, 1] - p[i, 1]
            distance = math.sqrt(dx * dx + dy * dy)
            if distance < rc_value and distance > 1e-12:
                scale = k_value * (distance - r0_value) / distance
                fx = scale * dx
                fy = scale * dy
                f[i, 0] += fx
                f[i, 1] += fy
                f[j, 0] -= fx
                f[j, 1] -= fy
    return f

def force_numpy(p, k_value, r0_value, rc_value):
    r = p[None, :, :] - p[:, None, :]
    distance = np.linalg.norm(r, axis=2)
    safe_distance = np.where(distance > 1e-12, distance, 1.0)
    magnitude = k_value * (distance - r0_value) / safe_distance
    pair_f = magnitude[:, :, None] * r
    mask = (distance < rc_value) & (distance > 1e-12)
    pair_f *= mask[:, :, None]
    return np.sum(pair_f, axis=1)

c_source = r"""#include <math.h>
void calculate_forces(int n, const double *p, double *f, double k, double r0, double rc) {
    for (int i = 0; i < 2*n; i++) f[i] = 0.0;
    for (int i = 0; i < n-1; i++) {
        for (int j = i+1; j < n; j++) {
            double dx = p[2*j] - p[2*i];
            double dy = p[2*j+1] - p[2*i+1];
            double r = sqrt(dx*dx + dy*dy);
            if (r < rc && r > 1e-12) {
                double scale = k * (r-r0) / r;
                double fx = scale * dx;
                double fy = scale * dy;
                f[2*i] += fx; f[2*i+1] += fy;
                f[2*j] -= fx; f[2*j+1] -= fy;
            }
        }
    }
}"""
with open("force_code.c", "w") as file:
    file.write(c_source)
subprocess.run(["gcc", "-O2", "-shared", "-fPIC", "force_code.c", "-o", "force_code.so"], check=True)
c_library = ctypes.CDLL(os.path.abspath("force_code.so"))
c_function = c_library.calculate_forces
c_function.argtypes = [ctypes.c_int, np.ctypeslib.ndpointer(dtype=np.float64),
                       np.ctypeslib.ndpointer(dtype=np.float64), ctypes.c_double,
                       ctypes.c_double, ctypes.c_double]

def force_c(p, k_value, r0_value, rc_value):
    p = np.ascontiguousarray(p, dtype=np.float64)
    f = np.zeros_like(p)
    c_function(len(p), p, f, k_value, r0_value, rc_value)
    return f

benchmark_N = [50, 200, 500, 1000]
benchmark_box = 1000.0
benchmark_rc = 100.0
benchmark_k = 15.0
benchmark_r0 = 40.0
benchmark_results = []

for n in benchmark_N:
    p = np.random.uniform(0, benchmark_box, (n, 2)).astype(np.float64)
    # Check that the implementations give the same answer
    f_loop = force_python_loop(p, benchmark_k, benchmark_r0, benchmark_rc)
    f_numpy = force_numpy(p, benchmark_k, benchmark_r0, benchmark_rc)
    f_compiled = force_c(p, benchmark_k, benchmark_r0, benchmark_rc)
    print("N =", n, "NumPy agrees:", np.allclose(f_loop, f_numpy, rtol=1e-8, atol=1e-8),
          "C agrees:", np.allclose(f_loop, f_compiled, rtol=1e-8, atol=1e-8))

    timings = []
    for method in [force_python_loop, force_numpy, force_c]:
        samples = []
        for repeat in range(5):
            start = time.perf_counter()
            method(p, benchmark_k, benchmark_r0, benchmark_rc)
            samples.append(time.perf_counter() - start)
        timings.append(float(np.median(samples)))
    benchmark_results.append((n, *timings))

print()
print("N | Python loop (s) | NumPy (s) | C (s)")
for row in benchmark_results:
    print(row)

In [ ]:
sizes = [row[0] for row in benchmark_results]
python_times = [row[1] for row in benchmark_results]
numpy_times = [row[2] for row in benchmark_results]
c_times = [row[3] for row in benchmark_results]

plt.figure(figsize=(8, 5))
plt.plot(sizes, python_times, marker="o", label="Python loop")
plt.plot(sizes, numpy_times, marker="o", label="NumPy")
plt.plot(sizes, c_times, marker="o", label="Compiled C")
plt.xlabel("Number of particles (N)")
plt.ylabel("Median wall time (seconds)")
plt.title("Force calculation time vs particle count")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(sizes, np.array(python_times) / np.array(numpy_times), marker="o", label="NumPy speedup")
plt.plot(sizes, np.array(python_times) / np.array(c_times), marker="o", label="C speedup")
plt.xlabel("Number of particles (N)")
plt.ylabel("Speedup over Python loop")
plt.title("Force calculation speedup")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Speedup table (relative to the Python loop) and scaling with N
print("N | Python (s) | NumPy (s) | C (s) | NumPy speedup | C speedup")
for row in benchmark_results:
    n, tp, tn, tc = row
    print(n, "|", format(tp, ".2e"), "|", format(tn, ".2e"), "|", format(tc, ".2e"), "|",
          format(tp / tn, ".1f"), "x |", format(tp / tc, ".1f"), "x")

# How does the time grow with N?  Fit  time ~ N^slope  on a log-log scale
log_sizes = np.log(sizes)
for name, values in [("Python loop", python_times), ("NumPy", numpy_times), ("C", c_times)]:
    slope = np.polyfit(log_sizes, np.log(values), 1)[0]
    print(name, ": time grows roughly like N^" + format(slope, ".2f"))

**Answer:** At N = 1000, NumPy is about 4 to 5 times faster than the Python loop and compiled C is roughly 200 times faster. The C speedup grows from about 50 to 60 times at N = 50 to over 160 times for N of 200 and above (see the table). Time grows roughly like N^2 for the Python loop and NumPy (fitted exponents close to 2) because every pair is checked, which is N(N-1)/2 pairs. The fitted exponent for C is lower (about 1.4 to 1.5) only because at small N its time is dominated by fixed call overhead. Its real cost is also proportional to N^2.

The **bottleneck** is the all-pairs distance calculation. Python pays interpreter overhead for every pair. NumPy removes the loop overhead but builds full N x N arrays, which costs memory (see Task 7). C does the same N^2 work with no loop overhead and no temporary arrays. Timings change a little from run to run and from computer to computer, so quote the values from your own final run.

# 18. Task 7 — Memory and scalability

A fully vectorized pair-displacement array has shape `(N, N, 2)`. Its memory use is `N * N * 2 * itemsize`. This estimate is for that one array only; a real calculation may create additional temporary arrays.

In [ ]:
memory_sizes = [100, 1000, 5000, 10000]
print("N | float32 MB | float64 MB")
for n in memory_sizes:
    memory32 = n * n * 2 * np.dtype(np.float32).itemsize / (1024 ** 2)
    memory64 = n * n * 2 * np.dtype(np.float64).itemsize / (1024 ** 2)
    print(n, "|", round(memory32, 2), "|", round(memory64, 2))

# Check NumPy's actual nbytes for a manageable example
n_check = 1000
displacement_check = np.zeros((n_check, n_check, 2), dtype=np.float64)
theory_bytes = n_check * n_check * 2 * np.dtype(np.float64).itemsize
print()
print("Theoretical bytes:", theory_bytes)
print("NumPy nbytes:", displacement_check.nbytes)
del displacement_check

In [ ]:
import tracemalloc

# Measure the real peak memory used by the NumPy force calculation (not just one array)
print("N | one (N,N,2) float64 array (MB) | peak memory of force_numpy (MB) | peak / one array")
peak_factor = 1.0
for n in [500, 1000, 2000, 3000]:
    p = np.random.uniform(0, 1000.0, (n, 2))
    tracemalloc.start()
    force_numpy(p, benchmark_k, benchmark_r0, benchmark_rc)
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    one_array = n * n * 2 * 8
    peak_factor = peak / one_array
    print(n, "|", round(one_array / 1024 ** 2, 1), "|", round(peak / 1024 ** 2, 1), "|", round(peak_factor, 1))

# Estimate where the vectorised method stops being practical.
# We call it impractical when its peak memory would use more than half of the computer's RAM.
my_ram_gb = 8          # <-- change this to the RAM of your computer
usable_bytes = 0.5 * my_ram_gb * 1024 ** 3
for name, itemsize in [("float64", 8), ("float32", 4)]:
    n_limit = int(np.sqrt(usable_bytes / (peak_factor * 2 * itemsize)))
    print(name, ": vectorised force becomes impractical at about N =", n_limit, "for", my_ram_gb, "GB of RAM")

**Answer:** The theoretical size of one (N, N, 2) array matches NumPy's `nbytes` exactly (16,000,000 bytes for N = 1000 in float64). The real force calculation needs more than one such array: the measured **peak memory was about 3.6 times** the size of one array, because of temporary arrays for distances, masks and pair forces.

We call the vectorised method impractical when its peak memory exceeds half of the computer's RAM. With 8 GB of RAM that happens at **about N = 8600 in float64** and about N = 12000 in float32 (set `my_ram_gb` to your own RAM and re-run). For example, at N = 10000 one float64 array alone is 1.5 GB and the peak would be about 5.5 GB. A method that is fast for small N is not necessarily scalable, because its memory grows like N^2.

# 19. Task 8 — Innovation: simple cell-linked neighbour search

**Innovation:** instead of checking every possible pair, divide the box into square cells whose side is at least the cutoff. Only particles in the same or neighbouring cells can interact. This reduces the number of distance checks for sparse systems. We compare its force result with the original Python all-pairs loop and measure the runtime. This is a simple cell-list implementation, not a replacement for the baseline.

In [ ]:
def force_cell_list(p, box_size, cutoff, k_value, r0_value):
    n = len(p)
    forces = np.zeros_like(p)
    number_cells = max(1, int(box_size / cutoff))
    cell_size = box_size / number_cells
    cells = {}

    # Put each particle into a cell
    for i in range(n):
        cx = min(int(p[i, 0] / cell_size), number_cells - 1)
        cy = min(int(p[i, 1] / cell_size), number_cells - 1)
        key = (cx, cy)
        if key not in cells:
            cells[key] = []
        cells[key].append(i)

    # Check particles in the same cell and neighbouring cells
    checked_pairs = set()
    for (cx, cy), particle_ids in cells.items():
        for dx in [-1, 0, 1]:
            for dy in [-1, 0, 1]:
                nx = cx + dx
                ny = cy + dy
                if nx < 0 or ny < 0 or nx >= number_cells or ny >= number_cells:
                    continue
                other_ids = cells.get((nx, ny), [])
                for i in particle_ids:
                    for j in other_ids:
                        if i < j and (i, j) not in checked_pairs:
                            checked_pairs.add((i, j))
                            r = p[j] - p[i]
                            distance = np.linalg.norm(r)
                            if distance < cutoff and distance > 1e-12:
                                force = k_value * (distance - r0_value) * r / distance
                                forces[i] += force
                                forces[j] -= force
    return forces

# Compare the all-pairs Python loop, the cell list, and compiled C for several N.
# The box grows with N so that the number of particles per area stays the same.
innovation_cutoff = 50.0
cell_results = []
print("N | all-pairs Python (s) | cell list (s) | cell-list speedup | all-pairs C (s) | same forces?")

for n in [200, 500, 1000, 2000]:
    box = 1000.0 * np.sqrt(n / 1000)
    p = np.random.uniform(0, box, (n, 2))
    old_force = force_python_loop(p, benchmark_k, benchmark_r0, innovation_cutoff)
    new_force = force_cell_list(p, box, innovation_cutoff, benchmark_k, benchmark_r0)
    agree = np.allclose(old_force, new_force)

    old_samples = []
    new_samples = []
    c_samples = []
    for repeat in range(5):
        start = time.perf_counter()
        force_python_loop(p, benchmark_k, benchmark_r0, innovation_cutoff)
        old_samples.append(time.perf_counter() - start)
        start = time.perf_counter()
        force_cell_list(p, box, innovation_cutoff, benchmark_k, benchmark_r0)
        new_samples.append(time.perf_counter() - start)
        start = time.perf_counter()
        force_c(p, benchmark_k, benchmark_r0, innovation_cutoff)
        c_samples.append(time.perf_counter() - start)

    old_median = np.median(old_samples)
    new_median = np.median(new_samples)
    c_median = np.median(c_samples)
    cell_results.append((n, old_median, new_median, c_median))
    print(n, "|", format(old_median, ".3e"), "|", format(new_median, ".3e"), "|",
          format(old_median / new_median, ".1f"), "x |", format(c_median, ".3e"), "|", agree)

**Answer:** The cell list gives the same forces as the all-pairs loop (every check prints True). It is faster than the Python all-pairs loop by about 1.6x at N = 200, 4x at N = 500, 7 to 8x at N = 1000 and about 15x at N = 2000. The speedup grows with N because the all-pairs cost grows like N^2, while the cell-list cost grows roughly like N, since each particle only looks at nearby cells.

Two honest limits: (1) compiled C all-pairs is still about 16x faster than our pure-Python cell list at N = 2000, so the cell list improves on the Python loop, not on C; combining a cell list with C would be the next step. (2) For the actual simulation (N = 12, L = 400, rc = 150) the box has only 2 x 2 cells, so the cell list gives no benefit there. Timings change a little from run to run, so quote the values from your own final run.

# 20. Task 9 — Final computational engineering decision

Every choice below is printed together with the number from this notebook that supports it.

In [ ]:
print("FINAL CONFIGURATION AND SUPPORTING NUMBERS")
print()
print("1. Precision: float64")
print("   float32 position difference:", position_error, " memory ratio:", mem64 / mem32, " time ratio:", time64 / time32)
print("2. Time step dt:", largest_dt)
print("   energy drift at this dt:", accuracy_results[largest_dt][4], "(tolerance", tolerance, ")")
print("3. Integration method: semi-implicit Euler (the baseline step_once)")
n, tp, tn, tc = benchmark_results[-1]
print("4. Force method: compiled C for large N")
print("   at N =", n, ": NumPy is", round(tp / tn, 1), "x and C is", round(tp / tc, 1), "x faster than the Python loop")
print("5. Cutoff radius: rc =", rc, "(baseline value, not varied in this project)")
print("6. Dissipation (when a static state is wanted): gamma = 0.5")
print("   relaxation time:", dissipation_results[0.5][5], " (gamma = 0.1:", dissipation_results[0.1][5], ")")
n, old_t, new_t, c_t = cell_results[-1]
print("7. Innovation: cell-linked neighbour list for large sparse systems")
print("   at N =", n, ": speedup over the Python all-pairs loop =", round(old_t / new_t, 1), "x;",
      "compiled C all-pairs is still", round(new_t / c_t, 1), "x faster than the Python cell list")

**Recommendation:** use **float64**, **dt = 0.01**, the **semi-implicit Euler** method, the **compiled C force routine** for large N (NumPy is fine up to a few thousand particles), **cutoff rc = 150**, **gamma = 0.5** when a static final state is wanted, and the **cell list** for large, sparse systems.

Why: float32 saves memory but gave no speed benefit at this size, so we keep the safer float64 and would switch to float32 only for very large N. dt = 0.01 is the largest step that meets our 5% drift tolerance (measured 3.2%). C was roughly 200x faster than the Python loop for N of 200 and above. gamma = 0.5 was the only value that reached our relaxation criterion within 300 time units (t = 149), so it is the practical choice when a static final state is wanted. The cell list was about 15x faster than the Python loop at N = 2000. The cutoff was not varied in this project, so rc = 150 is kept from the baseline and is not chosen from evidence. Re-check these numbers on the final computer.

# 21. Individual contributions and submission checklist

Fill this section with the actual work done by each person. Do not leave anyone without a contribution. Update names and roll numbers if the group changes.

| Group member | Roll number | Actual contribution | Tests/plots/video segment |
|---|---|---|---|
| Krishna Chaitanya Namala | ES24BTECH11023 | To be filled by group | To be filled |
| Rohith Perugu | ES24BTECH11026 | To be filled by group | To be filled |
| Purna Nanda Reddy Chintapally | ES24BTECH11014 | To be filled by group | To be filled |
| Member 4 | To be filled | To be filled | To be filled |

**Files produced by this notebook:** `particle_animation.gif` and the figures displayed above.

**Final submission checklist**
- [ ] Completed group details and individual contributions
- [ ] Executed notebook and any required data files
- [ ] Animation GIF or MP4
- [ ] PDF report with equations, methods, tables, plots and conclusions
- [ ] 15–20 minute video presentation with every member speaking
- [ ] Measured values checked on the computer used for the final submission